PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [8]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from rapidfuzz import process, fuzz
import pandas as pd
import unidecode
import re

In [10]:
player_stats_df= pd.read_csv('Data/final_player_stats.csv')

In [11]:
def nan_analysis(df):
    """
    This function takes a DataFrame and returns a DataFrame with the percentage of NaN values for each column.
    Only columns with NaN values are included in the output DataFrame.
    """
    nan_percentage = df.isna().mean() * 100
    nan_percentage = nan_percentage[nan_percentage > 0].reset_index()
    nan_percentage.columns = ['Column Name', 'Percentage of NaN Values']
    nan_percentage = nan_percentage.sort_values(by='Percentage of NaN Values', ascending=False)
    return nan_percentage

In [12]:
nan_analysis(player_stats_df)

,Column Name,Percentage of NaN Values
0,Marking,21.344515
1,Composure,10.848100
2,Stamina,0.013526
3,Sprint Speed,0.013526


In [13]:
player_stats_df

,Player Name,Team Name,Version,Ball Control,Dribbling,Marking,Slide Tackle,Stand Tackle,Aggression,Reactions,...,Curve,FK Acc.,Penalties,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Player URL
0,Adam Smith,AFC Bournemouth,fifa24,74.0,68.0,NaN,74.0,75.0,73.0,73.0,...,67.0,66.0,55.0,43.0,9.0,8.0,9.0,15.0,6.0,https://www.fifaindex.com/player/190885/adam-s...
1,Antoine Semenyo,AFC Bournemouth,fifa24,69.0,72.0,NaN,11.0,19.0,62.0,68.0,...,49.0,39.0,51.0,56.0,15.0,10.0,11.0,14.0,7.0,https://www.fifaindex.com/player/241236/antoin...
2,Chris Mepham,AFC Bournemouth,fifa24,64.0,58.0,NaN,72.0,74.0,73.0,68.0,...,41.0,35.0,44.0,33.0,14.0,10.0,10.0,12.0,12.0,https://www.fifaindex.com/player/233201/chris-...
3,Dango Ouattara,AFC Bournemouth,fifa24,74.0,78.0,NaN,44.0,51.0,46.0,66.0,...,66.0,55.0,66.0,49.0,14.0,9.0,6.0,13.0,10.0,https://www.fifaindex.com/player/265552/dango-...
4,Darren Randolph,AFC Bournemouth,fifa24,33.0,14.0,NaN,14.0,14.0,29.0,70.0,...,15.0,14.0,30.0,16.0,70.0,74.0,70.0,66.0,73.0,https://www.fifaindex.com/player/169697/darren...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7388,Rúben Neves,Wolves,fifa23,82.0,75.0,NaN,77.0,79.0,82.0,81.0,...,85.0,81.0,76.0,82.0,8.0,16.0,12.0,10.0,10.0,https://www.fifaindex.com/player/224293/r%C3%B...
7389,Saša Kalajdžić,Wolves,fifa23,78.0,74.0,NaN,47.0,46.0,44.0,78.0,...,50.0,47.0,57.0,73.0,14.0,7.0,6.0,13.0,15.0,https://www.fifaindex.com/player/239901/sa%C5%...
7390,Theo Corbeanu,Wolves,fifa23,66.0,71.0,NaN,45.0,49.0,40.0,57.0,...,64.0,64.0,55.0,50.0,10.0,7.0,6.0,9.0,7.0,https://www.fifaindex.com/player/258758/theo-c...
7391,Toti Gomes,Wolves,fifa23,62.0,58.0,NaN,73.0,73.0,72.0,72.0,...,48.0,48.0,45.0,42.0,5.0,11.0,6.0,12.0,10.0,https://www.fifaindex.com/player/258607/toti-g...


In [6]:
player_stats_df["Player URL"]

0       https://www.fifaindex.com/player/190885/adam-s...
1       https://www.fifaindex.com/player/241236/antoin...
2       https://www.fifaindex.com/player/233201/chris-...
3       https://www.fifaindex.com/player/265552/dango-...
4       https://www.fifaindex.com/player/169697/darren...
                              ...                        
7388    https://www.fifaindex.com/player/224293/r%C3%B...
7389    https://www.fifaindex.com/player/239901/sa%C5%...
7390    https://www.fifaindex.com/player/258758/theo-c...
7391    https://www.fifaindex.com/player/258607/toti-g...
7392    https://www.fifaindex.com/player/246420/j%C3%A...
Name: Player URL, Length: 7393, dtype: object

In [7]:

def extract_fifaindex_id(url):
    match = re.search(r'/player/(\d+)/', url)
    return match.group(1) if match else np.nan

player_stats_df['Player ID'] = player_stats_df['Player URL'].apply(extract_fifaindex_id)


In [112]:
# Ensure 'Version' sorts chronologically
version_order = ['fifa16', 'fifa17', 'fifa18', 'fifa19', 'fifa20', 'fifa21', 'fifa22', 'fifa23', 'fifa24']
player_stats_df['Version'] = pd.Categorical(player_stats_df['Version'], categories=version_order, ordered=True)

# Sort correctly by Player ID and Version
player_stats_df = player_stats_df.sort_values(by=['Player ID', 'Version'])

# For the latest available 'Player Name' per Player ID
player_name_map = (
    player_stats_df.groupby('Player ID')['Player Name']
    .apply(lambda x: x.dropna().iloc[-1])  # take the last available name
    .to_dict()
)

# Map it back
player_stats_df['Fifa_Name'] = player_stats_df['Player ID'].map(player_name_map)


In [113]:
player_stats_df.sort_values(by=['Player URL', 'Version'], inplace=True)

In [114]:
transfermarkt_df=pd.read_csv("/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Transfermarkt/Data/transfermarkt_player_stats_1524_cleaned.csv")

In [115]:
understat_roster_df=pd.read_csv("/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Understat/Data/understat_roster_1524.csv")

In [160]:
understat_roster_df

,Understat_Roster_ID,goals,own_goals,shots,xG,time,Understat_Player_ID,Understat_Team_ID,Understat_Position,Understat_Player_Name,...,xA,xGChain,xGBuildup,positionOrder,Understat_Match_Link,Best_Match_Name,Match_Score,Understat_Match_ID,id,season
0,620382,0,0,0,0.000000,90,560,89,GK,sergio romero,...,0.000000,0.000000,0.000000,1,https://understat.com/match/81,sergio romero,100.000000,81,81,2015
1,620383,0,0,0,0.000000,82,557,89,DR,matteo darmian,...,0.106513,0.106513,0.106513,2,https://understat.com/match/81,matteo darmian,100.000000,81,81,2015
2,620385,0,0,0,0.000000,90,628,89,DC,chris smalling,...,0.000000,0.106513,0.106513,3,https://understat.com/match/81,chris smalling,100.000000,81,81,2015
3,620384,0,0,0,0.000000,90,548,89,DC,daley blind,...,0.000000,0.127738,0.127738,3,https://understat.com/match/81,daley blind,100.000000,81,81,2015
4,620386,0,0,0,0.000000,90,1006,89,DL,luke shaw,...,0.000000,0.021225,0.021225,4,https://understat.com/match/81,luke shaw,100.000000,81,81,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96086,663140,0,0,1,0.113749,9,7198,82,Sub,oliver skipp,...,0.000000,0.113749,0.000000,17,https://understat.com/match/22274,oliver skipp,100.000000,22274,22274,2023
96087,663138,0,0,0,0.000000,9,7430,82,Sub,emerson,...,0.000000,0.067532,0.067532,17,https://understat.com/match/22274,emerson,100.000000,22274,22274,2023
96088,663141,0,0,0,0.000000,1,9332,82,Sub,dane scarlett,...,0.000000,0.000000,0.000000,17,https://understat.com/match/22274,dane scarlett,100.000000,22274,22274,2023
96089,663139,0,0,1,0.046925,19,343,82,Sub,pierre-emile hojbjerg,...,0.000000,0.901846,0.901846,17,https://understat.com/match/22274,pierre-emile hojbjerg,100.000000,22274,22274,2023


In [116]:
understat_roster_df.rename(columns={
    "id":"Understat_Roster_ID",
    "player_id":"Understat_Player_ID",
    "player":"Understat_Player_Name",
    "team_id":"Understat_Team_ID",
    "position":"Understat_Position",
    "match_link":"Understat_Match_Link"},
    inplace=True
)

In [117]:
def clean_name(name):
    return unidecode.unidecode(name.lower().replace(".", "").replace(",", "").strip())


In [118]:
player_stats_df["Fifa_Name"] = player_stats_df["Fifa_Name"].apply(clean_name)
transfermarkt_df["Transfermarkt_Name"] = transfermarkt_df["Transfermarkt_Name"].apply(clean_name)
understat_roster_df["Understat_Player_Name"]=understat_roster_df["Understat_Player_Name"].apply(clean_name)

In [119]:
# Create a lookup dictionary for best matches
fifa_names= player_stats_df['Fifa_Name'].dropna().unique()

def get_best_match(name):
    match, score, _ = process.extractOne(name, fifa_names, scorer=fuzz.WRatio)
    return pd.Series([match, score])

# Apply matching
understat_roster_df[['Best_Match_Name', 'Match_Score']] = understat_roster_df['Understat_Player_Name'].apply(get_best_match)

# Filter good matches (threshold can be adjusted)
exact_matched= understat_roster_df[understat_roster_df['Match_Score'] == 100]
matched = understat_roster_df[understat_roster_df['Match_Score'] >= 90]
almost_matched = understat_roster_df[(understat_roster_df['Match_Score'] >= 80) & (understat_roster_df['Match_Score'] < 90)]

not_matched= understat_roster_df[understat_roster_df['Match_Score'] < 80]


# Merge based on the fuzzy match name
merged = pd.merge(matched, player_stats_df, left_on='Best_Match_Name', right_on='Fifa_Name', how='left')


In [120]:
matched=matched.drop_duplicates(subset=["Understat_Player_Name",
                                        "Best_Match_Name"])[["Understat_Player_ID",
                                                             "Understat_Player_Name",
                                                             "Match_Score", 
                                                             "Best_Match_Name"]].sort_values(by="Match_Score")
matched=pd.merge(matched, player_stats_df, left_on='Best_Match_Name', right_on='Fifa_Name', how='left')
matched=matched[["Understat_Player_ID",
                 "Understat_Player_Name",
                 "Match_Score", 
                 "Best_Match_Name",
                 "Fifa_Name",
                 "Player URL"]]
matched=matched.drop_duplicates(subset=["Understat_Player_Name"])
matched=matched.sort_values(by="Match_Score", ascending=False)

In [121]:
to_be_merged=almost_matched.drop_duplicates(subset=["Understat_Player_Name",
                                        "Best_Match_Name"])[["Understat_Player_ID",
                                                             "Understat_Player_Name",
                                                             "Match_Score", 
                                                             "Best_Match_Name"]].sort_values(by="Match_Score")
merged_for_names = pd.merge(to_be_merged, player_stats_df, left_on='Best_Match_Name', right_on='Fifa_Name', how='left')
to_be_inspected=merged_for_names[["Understat_Player_ID",
                                  "Understat_Player_Name",
                                  "Match_Score", 
                                  "Best_Match_Name",
                                  "Fifa_Name",
                                  "Player URL"]]
to_be_inspected=to_be_inspected.drop_duplicates(subset=["Understat_Player_Name"])
to_be_inspected=to_be_inspected.sort_values(by="Match_Score", ascending=False)

In [122]:
to_be_inspected.to_excel("Data/inspect.xlsx")

In [123]:
inspected=pd.read_excel("Data/inspect_stated.xlsx")

In [124]:
inspected_true=inspected[inspected["State"] == True]
inspected_false=inspected[inspected["State"] == False]

In [125]:
matched=pd.concat([matched, inspected_true], ignore_index=True)

In [126]:
to_be_merged=not_matched.drop_duplicates(subset=["Understat_Player_Name",
                                        "Best_Match_Name"])[["Understat_Player_ID",
                                                             "Understat_Player_Name",
                                                             "Match_Score", 
                                                             "Best_Match_Name"]].sort_values(by="Match_Score")
merged_for_names = pd.merge(to_be_merged, player_stats_df, left_on='Best_Match_Name', right_on='Fifa_Name', how='left')
to_be_inspected=merged_for_names[["Understat_Player_ID",
                                  "Understat_Player_Name",
                                  "Match_Score", 
                                  "Best_Match_Name",
                                  "Fifa_Name",
                                  "Player URL"]]
to_be_inspected=to_be_inspected.drop_duplicates(subset=["Understat_Player_Name"])
to_be_inspected=to_be_inspected.sort_values(by="Match_Score", ascending=False)

In [127]:
to_be_inspected.to_excel("Data/inspect.xlsx")

In [128]:
inspected=pd.read_excel("Data/inspect_stated_2.xlsx")
inspected_true=inspected[inspected["State"] == True]
inspected_false_2=inspected[inspected["State"] == False]

In [129]:
matched=pd.concat([matched, inspected_true], ignore_index=True)

In [130]:
unmatched=pd.concat([inspected_false, inspected_false_2], ignore_index=True)

In [131]:
unmatched_names_list = unmatched["Understat_Player_Name"].unique().tolist()

In [132]:
unmatched_names_df=understat_roster_df[understat_roster_df["Understat_Player_Name"].isin(unmatched_names_list)]

In [133]:
# sort fifa names by name
# fifa names is an array
fifa_names_sorted= player_stats_df.sort_values(by='Fifa_Name')['Fifa_Name'].unique().tolist()

In [134]:
matched.drop(columns=['Best_Match_Name', 'Match_Score', 'Unnamed: 0', 'State'], inplace=True)

In [135]:
understat_epl1524 = pd.read_csv("/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Understat/Data/understat_epl1524_processed.csv")

In [136]:
understat_season = understat_epl1524[['id','season']]

In [137]:
understat_roster_df.columns

Index(['Understat_Roster_ID', 'goals', 'own_goals', 'shots', 'xG', 'time',
       'Understat_Player_ID', 'Understat_Team_ID', 'Understat_Position',
       'Understat_Player_Name', 'h_a', 'yellow_card', 'red_card', 'roster_in',
       'roster_out', 'key_passes', 'assists', 'xA', 'xGChain', 'xGBuildup',
       'positionOrder', 'Understat_Match_Link', 'Best_Match_Name',
       'Match_Score'],
      dtype='object')

In [138]:
def extract_understat_match_id(url):
    match = re.search(r'/match/(\d+)', url)
    return match.group(1) if match else np.nan

In [139]:
#apply the function to the 'Understat_Match_Link' column
understat_roster_df['Understat_Match_ID'] = understat_roster_df['Understat_Match_Link'].apply(extract_understat_match_id)

In [142]:
understat_roster_df['Understat_Match_ID'] = understat_roster_df['Understat_Match_ID'].astype(int)


In [143]:
understat_roster_df=pd.merge(understat_roster_df, understat_season, left_on="Understat_Match_ID", right_on="id", how='left')

In [144]:
understat_roster_df

,Understat_Roster_ID,goals,own_goals,shots,xG,time,Understat_Player_ID,Understat_Team_ID,Understat_Position,Understat_Player_Name,...,xA,xGChain,xGBuildup,positionOrder,Understat_Match_Link,Best_Match_Name,Match_Score,Understat_Match_ID,id,season
0,620382,0,0,0,0.000000,90,560,89,GK,sergio romero,...,0.000000,0.000000,0.000000,1,https://understat.com/match/81,sergio romero,100.000000,81,81,2015
1,620383,0,0,0,0.000000,82,557,89,DR,matteo darmian,...,0.106513,0.106513,0.106513,2,https://understat.com/match/81,matteo darmian,100.000000,81,81,2015
2,620385,0,0,0,0.000000,90,628,89,DC,chris smalling,...,0.000000,0.106513,0.106513,3,https://understat.com/match/81,chris smalling,100.000000,81,81,2015
3,620384,0,0,0,0.000000,90,548,89,DC,daley blind,...,0.000000,0.127738,0.127738,3,https://understat.com/match/81,daley blind,100.000000,81,81,2015
4,620386,0,0,0,0.000000,90,1006,89,DL,luke shaw,...,0.000000,0.021225,0.021225,4,https://understat.com/match/81,luke shaw,100.000000,81,81,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96086,663140,0,0,1,0.113749,9,7198,82,Sub,oliver skipp,...,0.000000,0.113749,0.000000,17,https://understat.com/match/22274,oliver skipp,100.000000,22274,22274,2023
96087,663138,0,0,0,0.000000,9,7430,82,Sub,emerson,...,0.000000,0.067532,0.067532,17,https://understat.com/match/22274,emerson,100.000000,22274,22274,2023
96088,663141,0,0,0,0.000000,1,9332,82,Sub,dane scarlett,...,0.000000,0.000000,0.000000,17,https://understat.com/match/22274,dane scarlett,100.000000,22274,22274,2023
96089,663139,0,0,1,0.046925,19,343,82,Sub,pierre-emile hojbjerg,...,0.000000,0.901846,0.901846,17,https://understat.com/match/22274,pierre-emile hojbjerg,100.000000,22274,22274,2023


In [151]:
season_index = {"fifa16":2015,
                "fifa17":2016,
                "fifa18":2017,
                "fifa19":2018,
                "fifa20":2019,
                "fifa21":2020,
                "fifa22":2021,
                "fifa23":2022,
                "fifa24":2023}

# Convert "Version" column of player_stats_df to numeric values based on the season_index mapping
player_stats_df['season'] = player_stats_df['Version'].map(season_index)

In [152]:
understat_fifa_merged = pd.merge(understat_roster_df, matched, on= "Understat_Player_Name", how='left')

In [153]:
understat_fifa_merged

,Understat_Roster_ID,goals,own_goals,shots,xG,time,Understat_Player_ID_x,Understat_Team_ID,Understat_Position,Understat_Player_Name,...,positionOrder,Understat_Match_Link,Best_Match_Name,Match_Score,Understat_Match_ID,id,season,Understat_Player_ID_y,Fifa_Name,Player URL
0,620382,0,0,0,0.000000,90,560,89,GK,sergio romero,...,1,https://understat.com/match/81,sergio romero,100.000000,81,81,2015,560.0,sergio romero,https://www.fifaindex.com/player/173373/sergio...
1,620383,0,0,0,0.000000,82,557,89,DR,matteo darmian,...,2,https://understat.com/match/81,matteo darmian,100.000000,81,81,2015,557.0,matteo darmian,https://www.fifaindex.com/player/184392/matteo...
2,620385,0,0,0,0.000000,90,628,89,DC,chris smalling,...,3,https://understat.com/match/81,chris smalling,100.000000,81,81,2015,628.0,chris smalling,https://www.fifaindex.com/player/189881/chris-...
3,620384,0,0,0,0.000000,90,548,89,DC,daley blind,...,3,https://understat.com/match/81,daley blind,100.000000,81,81,2015,548.0,daley blind,https://www.fifaindex.com/player/190815/daley-...
4,620386,0,0,0,0.000000,90,1006,89,DL,luke shaw,...,4,https://understat.com/match/81,luke shaw,100.000000,81,81,2015,1006.0,luke shaw,https://www.fifaindex.com/player/205988/luke-s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96086,663140,0,0,1,0.113749,9,7198,82,Sub,oliver skipp,...,17,https://understat.com/match/22274,oliver skipp,100.000000,22274,22274,2023,7198.0,oliver skipp,https://www.fifaindex.com/player/241042/oliver...
96087,663138,0,0,0,0.000000,9,7430,82,Sub,emerson,...,17,https://understat.com/match/22274,emerson,100.000000,22274,22274,2023,1245.0,emerson,https://www.fifaindex.com/player/210736/emerson/
96088,663141,0,0,0,0.000000,1,9332,82,Sub,dane scarlett,...,17,https://understat.com/match/22274,dane scarlett,100.000000,22274,22274,2023,9332.0,dane scarlett,https://www.fifaindex.com/player/261025/dane-s...
96089,663139,0,0,1,0.046925,19,343,82,Sub,pierre-emile hojbjerg,...,17,https://understat.com/match/22274,pierre-emile hojbjerg,100.000000,22274,22274,2023,343.0,pierre-emile hojbjerg,https://www.fifaindex.com/player/213648/pierre...


In [150]:
player_stats_df

,Player Name,Team Name,Version,Ball Control,Dribbling,Marking,Slide Tackle,Stand Tackle,Aggression,Reactions,...,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Player URL,Player ID,Fifa_Name,Season
2933,Rob Green,Huddersfield Town,fifa18,15.0,14.0,12.0,11.0,12.0,26.0,70.0,...,12.0,72.0,73.0,70.0,60.0,74.0,https://www.fifaindex.com/player/101880/rob-gr...,101880,rob green,2017
740,Artur Boruc,Bournemouth,fifa16,18.0,14.0,13.0,20.0,20.0,38.0,72.0,...,13.0,74.0,82.0,72.0,62.0,83.0,https://www.fifaindex.com/player/105846/artur-...,105846,artur boruc,2015
781,Artur Boruc,Bournemouth,fifa17,18.0,14.0,13.0,20.0,20.0,38.0,72.0,...,13.0,74.0,75.0,72.0,62.0,82.0,https://www.fifaindex.com/player/105846/artur-...,105846,artur boruc,2016
820,Artur Boruc,Bournemouth,fifa18,18.0,14.0,13.0,20.0,20.0,38.0,70.0,...,13.0,77.0,75.0,73.0,62.0,76.0,https://www.fifaindex.com/player/105846/artur-...,105846,artur boruc,2017
862,Artur Boruc,Bournemouth,fifa19,18.0,14.0,15.0,20.0,20.0,38.0,69.0,...,13.0,77.0,76.0,75.0,64.0,75.0,https://www.fifaindex.com/player/105846/artur-...,105846,artur boruc,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2413,Gareth Barry,Everton,fifa16,73.0,64.0,79.0,80.0,80.0,80.0,78.0,...,65.0,7.0,7.0,13.0,9.0,12.0,https://www.fifaindex.com/player/6826/gareth-b...,6826,gareth barry,2015
2459,Gareth Barry,Everton,fifa17,76.0,68.0,79.0,80.0,80.0,80.0,78.0,...,65.0,7.0,7.0,13.0,9.0,12.0,https://www.fifaindex.com/player/6826/gareth-b...,6826,gareth barry,2016
6709,Gareth Barry,West Bromwich Albion,fifa18,76.0,68.0,72.0,73.0,76.0,78.0,79.0,...,65.0,7.0,7.0,13.0,9.0,12.0,https://www.fifaindex.com/player/6826/gareth-b...,6826,gareth barry,2017
3169,Mark Schwarzer,Leicester City,fifa16,17.0,13.0,17.0,20.0,19.0,40.0,76.0,...,17.0,78.0,70.0,73.0,67.0,69.0,https://www.fifaindex.com/player/684/mark-schw...,684,mark schwarzer,2015


In [155]:
understat_fifa_merged=pd.merge(understat_fifa_merged, player_stats_df, on=["Fifa_Name","season"], how='left')

In [156]:
understat_fifa_merged

,Understat_Roster_ID,goals,own_goals,shots,xG,time,Understat_Player_ID_x,Understat_Team_ID,Understat_Position,Understat_Player_Name,...,Penalties,Volleys,GK Positioning,GK Diving,GK Handling,GK Kicking,GK Reflexes,Player URL_y,Player ID,Season
0,620382,0,0,0,0.000000,90,560,89,GK,sergio romero,...,16.0,17.0,73.0,76.0,67.0,81.0,82.0,https://www.fifaindex.com/player/173373/sergio...,173373,2015
1,620383,0,0,0,0.000000,82,557,89,DR,matteo darmian,...,45.0,59.0,12.0,6.0,7.0,7.0,7.0,https://www.fifaindex.com/player/184392/matteo...,184392,2015
2,620385,0,0,0,0.000000,90,628,89,DC,chris smalling,...,52.0,54.0,15.0,10.0,15.0,16.0,15.0,https://www.fifaindex.com/player/189881/chris-...,189881,2015
3,620384,0,0,0,0.000000,90,548,89,DC,daley blind,...,56.0,62.0,14.0,12.0,9.0,10.0,10.0,https://www.fifaindex.com/player/190815/daley-...,190815,2015
4,620386,0,0,0,0.000000,90,1006,89,DL,luke shaw,...,53.0,31.0,7.0,10.0,12.0,13.0,7.0,https://www.fifaindex.com/player/205988/luke-s...,205988,2015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97734,663140,0,0,1,0.113749,9,7198,82,Sub,oliver skipp,...,54.0,38.0,7.0,8.0,12.0,9.0,5.0,https://www.fifaindex.com/player/241042/oliver...,241042,2023
97735,663138,0,0,0,0.000000,9,7430,82,Sub,emerson,...,55.0,51.0,7.0,8.0,15.0,8.0,14.0,https://www.fifaindex.com/player/210736/emerson/,210736,2023
97736,663141,0,0,0,0.000000,1,9332,82,Sub,dane scarlett,...,67.0,59.0,5.0,12.0,8.0,7.0,14.0,https://www.fifaindex.com/player/261025/dane-s...,261025,2023
97737,663139,0,0,1,0.046925,19,343,82,Sub,pierre-emile hojbjerg,...,56.0,64.0,11.0,9.0,7.0,12.0,11.0,https://www.fifaindex.com/player/213648/pierre...,213648,2023


In [159]:
nan_analysis(understat_fifa_merged)

,Column Name,Percentage of NaN Values
8,Marking,25.000256
16,Composure,13.580045
21,Stamina,2.976294
24,Sprint Speed,2.976294
39,GK Reflexes,2.969132
38,GK Kicking,2.969132
25,Agility,2.969132
26,Jumping,2.969132
27,Heading,2.969132
28,Shot Power,2.969132


In [158]:
understat_fifa_merged.to_csv("Data/understat_fifa_merged.csv", index=False)